# Data sources 2/6: RSS, Atom and RDF feeds

`FeedReader` polls feeds with conditional requests, skips items it has already seen and
returns normalized `FeedItem`s. XML is parsed with `defusedxml`.

In [ ]:
from gitrecon.config import Config, load_env

load_env()  # GH_TOKEN / HF_TOKEN from dotenv files, never overriding your shell
config = Config()

In [ ]:
from gitrecon.sources.feeds import FeedReader

reader = FeedReader(state_file=None)  # no state file: every poll starts fresh
items = reader.poll("https://github.blog/changelog/feed/")
for item in items[:5]:
    print(f"{item.published:%Y-%m-%d} {item.title}  {item.categories}")

Polling again returns nothing new - the ids are remembered (and with a `state_file`,
also between runs):

In [ ]:
reader.poll("https://github.blog/changelog/feed/")

All feeds from the link catalog at once; a broken feed is reported, not fatal:

In [ ]:
from gitrecon.sources.links import load_catalog

catalog = config.data_dir / "catalog" / "links.jsonl"
urls = [link.url for link in load_catalog(catalog) if link.kind == "feed"] if catalog.exists() else []
for url, result in reader.poll_many(urls).items():
    print(f"{len(result) if isinstance(result, list) else result!r:>6}  {url}")

Parsing without the network - e.g. a file you already have:

In [ ]:
from gitrecon.sources.feeds import parse_feed

feed = parse_feed(b'''<feed xmlns="http://www.w3.org/2005/Atom"><title>demo</title>
<entry><id>1</id><title>Hello</title><updated>2026-10-03T00:00:00Z</updated></entry></feed>''')
feed.format, feed.title, feed.items[0].to_record()